# Pré-processamento de Dados - Modelo de Inferência do Perfil de Aprendizagem (LP)

Este notebook tem como objetivo realizar as etapas de pré-processamento dos dados simulados para o modelo de inferência do perfil de aprendizagem. Isso inclui codificação de variáveis categóricas, escalamento de variáveis numéricas e divisão dos dados em conjuntos de treino e teste.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import os

# Configurações para visualização (opcional, mas útil para verificar dados)
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 12

## 1. Carregamento dos Dados

In [4]:
data_path = os.path.join("..", "data", "raw", "simulated_lp_data.csv")
df = pd.read_csv(data_path)

print(f"Dataset carregado com {df.shape[0]} linhas e {df.shape[1]} colunas.")
df.head()

Dataset carregado com 2000 linhas e 13 colunas.


,time_spent_on_video,time_spent_on_audio,time_spent_reading,time_spent_writing,time_spent_on_quizz,time_spent_on_flashcards,completed_exercices,completed_quizzes,completed_flashcards,text_quizzes_accuracy,visual_quizzes_accuracy,most_preferred_resource_type,perfil_aprendizagem
0,74.901425,37.648523,39.777724,13.288780,19.669747,9.572042,41.0,17,73,68.096619,83.548181,audio,Auditivo
1,55.852071,61.931084,68.907878,20.536037,14.963497,12.556004,12.0,14,17,57.038022,85.903622,audio,Auditivo
2,126.794772,24.151602,70.630591,15.869097,18.276250,10.220952,40.0,2,23,80.712831,87.624325,video,Visual
3,152.720350,33.840769,86.542062,21.780063,27.147317,15.883344,5.0,6,0,58.785474,80.157925,video,Visual
4,81.199253,2.127707,22.159957,26.787745,32.778567,24.577428,46.0,5,93,67.497412,100.000000,video,Visual


## 2. Codificação do Target (`perfil_aprendizagem`)

In [5]:
le = LabelEncoder()
df["perfil_aprendizagem_encoded"] = le.fit_transform(df["perfil_aprendizagem"])

print("Mapeamento do LabelEncoder:")
for i, profile in enumerate(le.classes_):
    print(f"{profile}: {i}")

print("\nDistribuição do target codificado:")
print(df["perfil_aprendizagem_encoded"].value_counts())
df.head()

Mapeamento do LabelEncoder:
Auditivo: 0
Cinestésico: 1
Leitura/Escrita: 2
Visual: 3

Distribuição do target codificado:
perfil_aprendizagem_encoded
0    515
3    509
1    490
2    486
Name: count, dtype: int64


,time_spent_on_video,time_spent_on_audio,time_spent_reading,time_spent_writing,time_spent_on_quizz,time_spent_on_flashcards,completed_exercices,completed_quizzes,completed_flashcards,text_quizzes_accuracy,visual_quizzes_accuracy,most_preferred_resource_type,perfil_aprendizagem,perfil_aprendizagem_encoded
0,74.901425,37.648523,39.777724,13.288780,19.669747,9.572042,41.0,17,73,68.096619,83.548181,audio,Auditivo,0
1,55.852071,61.931084,68.907878,20.536037,14.963497,12.556004,12.0,14,17,57.038022,85.903622,audio,Auditivo,0
2,126.794772,24.151602,70.630591,15.869097,18.276250,10.220952,40.0,2,23,80.712831,87.624325,video,Visual,3
3,152.720350,33.840769,86.542062,21.780063,27.147317,15.883344,5.0,6,0,58.785474,80.157925,video,Visual,3
4,81.199253,2.127707,22.159957,26.787745,32.778567,24.577428,46.0,5,93,67.497412,100.000000,video,Visual,3


## 3. Codificação da Feature Categórica (`most_preferred_resource_type`)

In [6]:
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
encoded_features = ohe.fit_transform(df[["most_preferred_resource_type"]])
encoded_df = pd.DataFrame(encoded_features, columns=ohe.get_feature_names_out(["most_preferred_resource_type"]))

df = pd.concat([df, encoded_df], axis=1)
df = df.drop("most_preferred_resource_type", axis=1)

print("Features categóricas codificadas:")
df.head()

Features categóricas codificadas:


,time_spent_on_video,time_spent_on_audio,time_spent_reading,time_spent_writing,time_spent_on_quizz,time_spent_on_flashcards,completed_exercices,completed_quizzes,completed_flashcards,text_quizzes_accuracy,visual_quizzes_accuracy,perfil_aprendizagem,perfil_aprendizagem_encoded,most_preferred_resource_type_audio,most_preferred_resource_type_practical,most_preferred_resource_type_text,most_preferred_resource_type_video
0,74.901425,37.648523,39.777724,13.288780,19.669747,9.572042,41.0,17,73,68.096619,83.548181,Auditivo,0,1.0,0.0,0.0,0.0
1,55.852071,61.931084,68.907878,20.536037,14.963497,12.556004,12.0,14,17,57.038022,85.903622,Auditivo,0,1.0,0.0,0.0,0.0
2,126.794772,24.151602,70.630591,15.869097,18.276250,10.220952,40.0,2,23,80.712831,87.624325,Visual,3,0.0,0.0,0.0,1.0
3,152.720350,33.840769,86.542062,21.780063,27.147317,15.883344,5.0,6,0,58.785474,80.157925,Visual,3,0.0,0.0,0.0,1.0
4,81.199253,2.127707,22.159957,26.787745,32.778567,24.577428,46.0,5,93,67.497412,100.000000,Visual,3,0.0,0.0,0.0,1.0


## 4. Escalamento das Features Numéricas

In [7]:
numerical_features = df.select_dtypes(include=np.number).columns.tolist()
# Remover o target codificado da lista de features numéricas para escalamento
numerical_features.remove("perfil_aprendizagem_encoded")

scaler = StandardScaler()
df[numerical_features] = scaler.fit_transform(df[numerical_features])

print("Features numéricas escaladas:")
df.head()

Features numéricas escaladas:


,time_spent_on_video,time_spent_on_audio,time_spent_reading,time_spent_writing,time_spent_on_quizz,time_spent_on_flashcards,completed_exercices,completed_quizzes,completed_flashcards,text_quizzes_accuracy,visual_quizzes_accuracy,perfil_aprendizagem,perfil_aprendizagem_encoded,most_preferred_resource_type_audio,most_preferred_resource_type_practical,most_preferred_resource_type_text,most_preferred_resource_type_video
0,0.155811,-0.307935,-0.878355,-1.087956,-0.240826,-0.698104,0.478027,0.311498,0.791720,-0.825773,0.149532,Auditivo,0,1.698086,-0.569652,-0.566572,-0.584279
1,-0.368781,0.694114,-0.187535,-0.702923,-0.623374,-0.308357,-0.828272,-0.038697,-1.131849,-1.868375,0.377194,Auditivo,0,1.698086,-0.569652,-0.566572,-0.584279
2,1.584881,-0.864902,-0.146681,-0.950868,-0.354097,-0.613348,0.432982,-1.439476,-0.925752,0.363680,0.543506,Visual,3,-0.588898,-0.569652,-0.566572,1.711512
3,2.298835,-0.465067,0.230658,-0.636830,0.366988,0.126240,-1.143585,-0.972550,-1.715789,-1.703626,-0.178149,Visual,3,-0.588898,-0.569652,-0.566572,1.711512
4,0.329245,-1.773744,-1.296160,-0.370782,0.824725,1.261809,0.703251,-1.089281,1.478709,-0.882266,1.739659,Visual,3,-0.588898,-0.569652,-0.566572,1.711512


## 5. Divisão dos Dados em Conjuntos de Treino e Teste

In [8]:
X = df.drop(["perfil_aprendizagem", "perfil_aprendizagem_encoded"], axis=1)
y = df["perfil_aprendizagem_encoded"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print(f"Shape de X_train: {X_train.shape}")
print(f"Shape de X_test: {X_test.shape}")
print(f"Shape de y_train: {y_train.shape}")
print(f"Shape de y_test: {y_test.shape}")

print("\nDistribuição do target em y_train:")
print(y_train.value_counts(normalize=True) * 100)
print("\nDistribuição do target em y_test:")
print(y_test.value_counts(normalize=True) * 100)

Shape de X_train: (1400, 15)
Shape de X_test: (600, 15)
Shape de y_train: (1400,)
Shape de y_test: (600,)

Distribuição do target em y_train:
perfil_aprendizagem_encoded
0    25.785714
3    25.428571
1    24.500000
2    24.285714
Name: proportion, dtype: float64

Distribuição do target em y_test:
perfil_aprendizagem_encoded
0    25.666667
3    25.500000
1    24.500000
2    24.333333
Name: proportion, dtype: float64


## 6. Salvar Dados Pré-processados

In [10]:
output_dir = os.path.join("..", "data", "processed")
os.makedirs(output_dir, exist_ok=True)

X_train.to_csv(os.path.join(output_dir, "X_train.csv"), index=False)
X_test.to_csv(os.path.join(output_dir, "X_test.csv"), index=False)
y_train.to_csv(os.path.join(output_dir, "y_train.csv"), index=False)
y_test.to_csv(os.path.join(output_dir, "y_test.csv"), index=False)

print(f"Dados de treino e teste salvos em {output_dir}")

Dados de treino e teste salvos em ../data/processed
